In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()   # a file picker will appear: choose india_sample.csv

df = pd.read_csv("india_sample.csv")
print("Rows and columns:", df.shape)
print(df.columns.tolist())
df.head()

Saving india_sample.csv to india_sample.csv
Rows and columns: (899, 8)
['code', 'product_name', 'brands', 'categories_tags', 'allergens_tags', 'traces_tags', 'ingredients_text', 'states_tags']


,code,product_name,brands,categories_tags,allergens_tags,traces_tags,ingredients_text,states_tags
0,8901719134845,Parle-G Biscuit,Parle,en:snacks;en:sweet-snacks;en:biscuits-and-cake...,en:gluten;en:milk,NaN,"REFINED WHEAT FLOUR (MAIDA) 68%, SUGAR, REFINE...",en:to-be-completed;en:nutrition-facts-complete...
1,8901719134852,Parle-G,Parle,en:snacks;en:sweet-snacks;en:biscuits-and-cake...,NaN,NaN,NaN,en:to-be-completed;en:nutrition-facts-complete...
2,8901063139329,bourbon,britannia,en:snacks;en:sweet-snacks;en:biscuits-and-cake...,en:gluten;en:milk,NaN,Flavoured\nSandwich BiscuitsINGREDIENTS: REFIN...,en:to-be-completed;en:nutrition-facts-complete...
3,8902080000227,Sting Energy,Sting,en:beverages-and-beverages-preparations;en:bev...,en:Acesulfame Potassium,NaN,"CARBONATED WATER, SUGAR, ACIDITY REGULATORS (3...",en:to-be-completed;en:nutrition-facts-complete...
4,8906010500764,balaji wafers chataka pataka,Balaji,en:snacks;en:salty-snacks;en:sweet-snacks;en:a...,NaN,NaN,"Rice meal (43%), edible vegetable oil (palmole...",en:to-be-completed;en:nutrition-facts-complete...


In [ ]:
import pandas as pd

# The 14 major allergens in Open Food Facts' standard tag format
STANDARD = {"milk", "gluten", "eggs", "nuts", "peanuts", "soybeans", "fish",
            "crustaceans", "molluscs", "celery", "mustard", "sesame-seeds",
            "sulphur-dioxide-and-sulphites", "lupin"}

def split_tags(value):
    """'en:gluten;en:milk' -> ['gluten', 'milk']; empty -> []"""
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [t.strip().replace("en:", "").lower() for t in str(value).split(";") if t.strip()]

clean = df.copy()
clean["allergens"] = clean["allergens_tags"].apply(split_tags)
clean["traces"] = clean["traces_tags"].apply(split_tags)
clean["categories"] = clean["categories_tags"].apply(split_tags)

# Completeness flags (empty = "none listed OR nobody filled it in")
clean["has_allergen_data"] = clean["allergens"].str.len() > 0
clean["has_traces_data"] = clean["traces"].str.len() > 0
clean["has_ingredients"] = clean["ingredients_text"].notna()

# Broad category = first category tag
clean["main_category"] = clean["categories"].apply(lambda c: c[0] if c else "unknown")

# Flag entries with non-standard allergen values (like 'acesulfame potassium')
clean["has_nonstandard"] = clean["allergens"].apply(lambda a: any(x not in STANDARD for x in a))

# Quick report
print("Products:", len(clean))
print("With allergen data:  ", round(clean["has_allergen_data"].mean()*100, 1), "%")
print("With traces data:    ", round(clean["has_traces_data"].mean()*100, 1), "%")
print("With ingredients:    ", round(clean["has_ingredients"].mean()*100, 1), "%")
print("Non-standard allergen entries:", int(clean["has_nonstandard"].sum()))
print("\nTop 10 allergens:")
print(clean["allergens"].explode().value_counts().head(10))
print("\nTop 10 main categories:")
print(clean["main_category"].value_counts().head(10))
print("\nDuplicate product names:", clean["product_name"].duplicated().sum())

Products: 899
With allergen data:   42.0 %
With traces data:     8.2 %
With ingredients:     72.7 %
Non-standard allergen entries: 22

Top 10 allergens:
allergens
milk                             214
gluten                           195
soybeans                         135
nuts                              56
peanuts                           48
sesame-seeds                      37
sulphur-dioxide-and-sulphites     27
mustard                           10
apple                              6
orange                             4
Name: count, dtype: int64

Top 10 main categories:
main_category
snacks                                  291
plant-based-foods-and-beverages         221
beverages-and-beverages-preparations    168
unknown                                  72
dairies                                  37
condiments                               17
desserts                                 13
breakfasts                               11
dietary-supplements                       7
sweete

In [ ]:
import sqlite3

# 1) Cleaner category groups (approximate, keyword-based)
RULES = [
    ("Biscuits & cookies",   ["biscuit", "cookie"]),
    ("Chocolate & sweets",   ["chocolate", "confectioner", "candies", "candy"]),
    ("Noodles & pasta",      ["noodle", "pasta"]),
    ("Salty snacks",         ["salty-snacks", "chips", "crisps", "namkeen"]),
    ("Dairy",                ["dairies", "yogurt", "cheese"]),
    ("Beverages",            ["beverages-and-beverages-preparations"]),
    ("Cereals & breakfast",  ["cereal", "breakfast"]),
    ("Condiments & sauces",  ["condiment", "sauce", "spice"]),
    ("Other snacks",         ["snacks"]),
    ("Plant-based staples",  ["plant-based"]),
]

def category_group(cats):
    s = " ".join(cats)
    for name, keys in RULES:
        if any(k in s for k in keys):
            return name
    return "Other / unknown"

clean["category_group"] = clean["categories"].apply(category_group)
print(clean["category_group"].value_counts())

# 2) Build tables
products = clean[["code", "product_name", "brands", "category_group",
                  "has_allergen_data", "has_traces_data",
                  "has_ingredients", "has_nonstandard"]].copy()
for c in ["has_allergen_data", "has_traces_data", "has_ingredients", "has_nonstandard"]:
    products[c] = products[c].astype(int)

allergens = (clean[["code", "allergens"]].explode("allergens").dropna()
             .rename(columns={"allergens": "allergen"}))
allergens["is_standard"] = allergens["allergen"].isin(STANDARD).astype(int)

traces = (clean[["code", "traces"]].explode("traces").dropna()
          .rename(columns={"traces": "trace"}))

con = sqlite3.connect("allergen.db")
products.to_sql("products", con, if_exists="replace", index=False)
allergens.to_sql("allergens", con, if_exists="replace", index=False)
traces.to_sql("traces", con, if_exists="replace", index=False)

def run(sql):
    return pd.read_sql(sql, con)

# 3) Queries
print("\nQ1. Data completeness overall")
display(run("""
SELECT COUNT(*) AS products,
       ROUND(100.0*SUM(has_allergen_data)/COUNT(*),1) AS pct_with_allergens,
       ROUND(100.0*SUM(has_traces_data)/COUNT(*),1)   AS pct_with_traces,
       ROUND(100.0*SUM(has_ingredients)/COUNT(*),1)   AS pct_with_ingredients
FROM products"""))

print("Q2. Most common allergens (% of products that list allergens)")
display(run("""
SELECT a.allergen, COUNT(DISTINCT a.code) AS products,
       ROUND(100.0*COUNT(DISTINCT a.code) /
         (SELECT SUM(has_allergen_data) FROM products),1) AS pct_of_labeled
FROM allergens a WHERE a.is_standard = 1
GROUP BY a.allergen ORDER BY products DESC"""))

print("Q3. Data completeness by category")
display(run("""
SELECT category_group, COUNT(*) AS products,
       ROUND(100.0*SUM(has_allergen_data)/COUNT(*),1) AS pct_with_allergens
FROM products GROUP BY category_group ORDER BY products DESC"""))

print("Q4. Milk allergen share by category (labeled products only)")
display(run("""
SELECT p.category_group, COUNT(*) AS labeled_products,
       ROUND(100.0*SUM(CASE WHEN EXISTS (SELECT 1 FROM allergens a
            WHERE a.code=p.code AND a.allergen='milk') THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_milk
FROM products p WHERE p.has_allergen_data=1
GROUP BY p.category_group HAVING labeled_products >= 10
ORDER BY pct_milk DESC"""))

print("Q5. Nut or peanut share by category (labeled products only)")
display(run("""
SELECT p.category_group, COUNT(*) AS labeled_products,
       ROUND(100.0*SUM(CASE WHEN EXISTS (SELECT 1 FROM allergens a
            WHERE a.code=p.code AND a.allergen IN ('nuts','peanuts')) THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_nuts_or_peanuts
FROM products p WHERE p.has_allergen_data=1
GROUP BY p.category_group HAVING labeled_products >= 10
ORDER BY pct_nuts_or_peanuts DESC"""))

print("Q6. How many allergens per labeled product")
display(run("""
SELECT n AS allergens_listed, COUNT(*) AS products FROM
 (SELECT code, COUNT(*) AS n FROM allergens WHERE is_standard=1 GROUP BY code)
GROUP BY n ORDER BY n"""))

print("Q7. Most common 'may contain traces' warnings")
display(run("""
SELECT trace, COUNT(*) AS products FROM traces
GROUP BY trace ORDER BY products DESC LIMIT 10"""))

print("Q8. Non-standard allergen entries (data-quality issue)")
display(run("""
SELECT allergen, COUNT(*) AS times FROM allergens
WHERE is_standard=0 GROUP BY allergen ORDER BY times DESC LIMIT 15"""))

category_group
Biscuits & cookies     161
Beverages              157
Other / unknown        142
Salty snacks            98
Cereals & breakfast     63
Chocolate & sweets      60
Plant-based staples     56
Dairy                   46
Condiments & sauces     45
Other snacks            41
Noodles & pasta         30
Name: count, dtype: int64

Q1. Data completeness overall


,products,pct_with_allergens,pct_with_traces,pct_with_ingredients
0,899,42.0,8.2,72.7


Q2. Most common allergens (% of products that list allergens)


,allergen,products,pct_of_labeled
0,milk,214,56.6
1,gluten,195,51.6
2,soybeans,135,35.7
3,nuts,56,14.8
4,peanuts,48,12.7
5,sesame-seeds,37,9.8
6,sulphur-dioxide-and-sulphites,27,7.1
7,mustard,10,2.6
8,eggs,4,1.1
9,celery,3,0.8


Q3. Data completeness by category


,category_group,products,pct_with_allergens
0,Biscuits & cookies,161,64.0
1,Beverages,157,8.9
2,Other / unknown,142,19.7
3,Salty snacks,98,51.0
4,Cereals & breakfast,63,52.4
5,Chocolate & sweets,60,55.0
6,Plant-based staples,56,28.6
7,Dairy,46,78.3
8,Condiments & sauces,45,28.9
9,Other snacks,41,65.9


... (6 outputs left out)

Q7. Most common 'may contain traces' warnings


,trace,products
0,milk,43
1,gluten,34
2,nuts,33
3,soybeans,32
4,mustard,15
5,sesame-seeds,12
6,peanuts,12
7,sulphur-dioxide-and-sulphites,8
8,mustard seed,4
9,celery,3


Q8. Non-standard allergen entries (data-quality issue)


,allergen,times
0,apple,6
1,orange,4
2,none,3
3,steviol glycoside,1
4,soy & nut.,1
5,permitted antioxidants,1
6,peach,1
7,mustard seed,1
8,fruit,1
9,contains wheat,1


In [ ]:
RULES = [
    ("Biscuits & cookies",   ["biscuit", "cookie"]),
    ("Chocolate & sweets",   ["chocolate", "confectioner", "candies", "candy"]),
    ("Noodles & pasta",      ["noodle", "pasta"]),
    ("Salty snacks",         ["salty-snacks", "chips", "crisps", "namkeen"]),
    ("Dairy",                ["dairies", "yogurt", "cheese"]),
    ("Beverages",            ["beverages-and-beverages-preparations"]),
    ("Cereals & breakfast",  ["cereal", "breakfast"]),
    ("Condiments & sauces",  ["condiment", "sauce", "spice"]),
    ("Other snacks",         ["snacks"]),
    ("Plant-based staples",  ["plant-based"]),
]

In [ ]:
from google.colab import files

major = ["milk", "gluten", "soybeans", "nuts", "peanuts", "sesame-seeds",
         "sulphur-dioxide-and-sulphites", "mustard", "eggs"]

export = products.copy()
for a in major:
    export["has_" + a.replace("-", "_")] = clean["allergens"].apply(lambda x: int(a in x)).values
export["allergen_data_status"] = export["has_allergen_data"].map({1: "Allergens listed", 0: "No allergen data"})

export.to_csv("products_clean.csv", index=False)
allergens.to_csv("allergens_long.csv", index=False)
traces.to_csv("traces_long.csv", index=False)

print(export.shape)
for f in ["products_clean.csv", "allergens_long.csv", "traces_long.csv"]:
    files.download(f)

(899, 18)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Spot check: which products are in the groups with the highest nut share?
check = clean[clean["has_allergen_data"] & clean["allergens"].apply(lambda a: "nuts" in a or "peanuts" in a)]
print(check.groupby("category_group").size())
print()
print(check[["product_name", "category_group"]].sample(15, random_state=1).to_string())

category_group
Beverages              28
Biscuits & cookies     12
Cereals & breakfast     2
Chocolate & sweets      4
Condiments & sauces     1
Dairy                   1
Other / unknown         7
Other snacks           16
Salty snacks           15
dtype: int64

                          product_name       category_group
470                     salted peanuts            Beverages
499         Salt-n-Pepper Banana Chips         Salty snacks
339                      Farali Chiwda         Salty snacks
473        Tata Soulfull Millet Muesli            Beverages
458              Haldiram's Bhujia sev         Salty snacks
864       Hazelnut Choco Peanut spread   Chocolate & sweets
632         Amul Fruit 'N' Nut Fantasy      Other / unknown
748                       Amul Rajbhog      Other / unknown
892     Too Yumm! Namkeen - All In One         Other snacks
604               Chocos Crunchy Bites   Chocolate & sweets
718              Breakfast protein bar  Cereals & breakfast
562              

In [ ]:
from google.colab import files

major = ["milk", "gluten", "soybeans", "nuts", "peanuts", "sesame-seeds",
         "sulphur-dioxide-and-sulphites", "mustard", "eggs"]

export = products.copy()
for a in major:
    export["has_" + a.replace("-", "_")] = clean["allergens"].apply(lambda x: int(a in x)).values
export["allergen_data_status"] = export["has_allergen_data"].map({1: "Allergens listed", 0: "No allergen data"})

export.to_csv("products_clean.csv", index=False)
allergens.to_csv("allergens_long.csv", index=False)
traces.to_csv("traces_long.csv", index=False)

print(export.shape)
for f in ["products_clean.csv", "allergens_long.csv", "traces_long.csv"]:
    files.download(f)

(899, 18)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>